# Phase 1: Target Definition & Baseline
## Step 1.1: 신제품(NPD) 식별 및 출시 주차별 누적 매출 분포 분석

- **목표**: 각 상품의 최초 출시일을 정의하고, 출시 후 1, 2, 4주 차의 누적 매출액 분포를 통해 '성공'의 임계값($y$)을 도출합니다.
- **데이터**: `B2_POS_SALE.parquet` (전처리 완료된 POS 판매 데이터)
- **표준 컬럼명**: `판매일자`, `상품코드`, `판매금액` 등

In [16]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import sys
from scipy.stats import spearmanr

In [2]:
# 데이터 경로 설정
B2_PATH = '../../data/processed/B2_POS_SALE.parquet'
B4_PATH = '../../data/processed/B4_ITEM_DV_INFO.parquet'

# 1. 데이터 로드
def load_b2(path):
    return pl.scan_parquet(path)

b2_lazy = load_b2(B2_PATH)

In [6]:
# 2. 데이터 구조 및 샘플 확인
print("\n--- B2 데이터 샘플 (상위 5개 행) ---")
display(b2_lazy.head(5).collect())


--- B2 데이터 샘플 (상위 5개 행) ---


판매일자,판매시간,점포코드,POS번호,거래번호,상품코드,판매수량,판매금액
str,str,str,str,str,str,i64,i64
"""20250523""","""084531""","""64139""","""02""","""26651""","""314952""",1,500
"""20250523""","""105724""","""52657""","""01""","""48727""","""314954""",10,45000
"""20250523""","""105724""","""52657""","""01""","""48727""","""314952""",10,5000
"""20250523""","""084531""","""64139""","""02""","""26651""","""314983""",1,4500
"""20250523""","""083823""","""50134""","""01""","""67971""","""314988""",1,4500


## Data Integrity Check (데이터 무결성 검토)

In [8]:
print("--- [검토 1] 결측치 및 데이터 다양성 점검 ---")
# pl.all().null_count().suffix("_null") 대신 .name.suffix() 사용
null_and_unique = b2_lazy.select([
    pl.all().null_count().name.suffix("_null"),
    pl.all().n_unique().name.suffix("_unique")
]).collect()
display(null_and_unique)

--- [검토 1] 결측치 및 데이터 다양성 점검 ---


판매일자_null,판매시간_null,점포코드_null,POS번호_null,거래번호_null,상품코드_null,판매수량_null,판매금액_null,판매일자_unique,판매시간_unique,점포코드_unique,POS번호_unique,거래번호_unique,상품코드_unique,판매수량_unique,판매금액_unique
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,181,86400,448,22,99999,12603,540,20395


In [11]:
print("\n--- [검토 2] 판매수량 및 판매금액 이상치 점검 (음수 매출 확인) ---")
numeric_stats = b2_lazy.select([
    pl.col("판매수량").min().alias("수량_최소"),
    pl.col("판매수량").max().alias("수량_최대"),
    (pl.col("판매수량") < 0).sum().alias("수량_0미만_건수"),
    pl.col("판매금액").min().alias("금액_최소"),
    pl.col("판매금액").max().alias("금액_최대"),
    (pl.col("판매금액") < 0).sum().alias("금액_0미만_건수")
]).collect()
display(numeric_stats)


--- [검토 2] 판매수량 및 판매금액 이상치 점검 (음수 매출 확인) ---


수량_최소,수량_최대,수량_0미만_건수,금액_최소,금액_최대,금액_0미만_건수
i64,i64,u32,i64,i64,u32
-2000,2000,1056896,-1300000000,1300000000,1030054


In [10]:
print("\n--- [검토 3] 날짜/시간 형식 점검 (길이 확인) ---")
format_check = b2_lazy.select([
    (pl.col("판매일자").str.len_chars() != 8).sum().alias("일자_형식오류_건수"),
    (pl.col("판매시간").str.len_chars() != 6).sum().alias("시간_형식오류_건수")
]).collect()
display(format_check)


--- [검토 3] 날짜/시간 형식 점검 (길이 확인) ---


일자_형식오류_건수,시간_형식오류_건수
u32,u32
0,0


In [14]:
"""
[분석 가설 및 검증 로직: 음수(-) 매출 데이터의 정체 규명]

1. 배경
   - B2 POS 데이터에서 약 103만 건의 음수(-) 판매금액 데이터가 발견됨.
   - 최대/최소 금액이 대칭(+13억, -13억)을 이루는 등 시스템적 취소 노이즈 가능성 농후.

2. 분석 가설
   - 가설 A (시스템적 즉시 취소): 점포코드, POS번호, 거래번호, 상품코드, 금액(절대값)이 동일한 양수/음수 쌍은 
     오결제 등에 의한 즉시 취소(Void) 데이터이며, 이는 실질 매출에 영향을 주지 않는 노이즈이다.
   - 가설 B (소비자 변심 반품): 양수 거래 발생 후 일정 시간(예: 수 시간~수 일) 후에 발생하는 음수 거래는 
     실제 반품이며, 신제품의 성과 측정 시 이를 상계 처리(Net)해야 한다.

3. 검증 방법론
   - 양수(+) 데이터셋과 음수(-) 데이터셋을 분리한 후, 
     ['점포코드', 'POS번호', '거래번호', '상품코드']를 키로 Inner Join을 수행하여 매칭 쌍을 탐색한다.
   - 매칭된 쌍의 '판매시간' 차이를 계산하여 즉시 취소와 사후 반품의 비중을 산출한다.
"""

def validate_negative_data(b2_lazy):
    print("--- [검증 시작] 음수 트랜잭션의 정체 및 정당성 분석 ---")

    # 1. 데이터 분리
    pos_sales = b2_lazy.filter(pl.col("판매금액") > 0)
    neg_sales = b2_lazy.filter(pl.col("판매금액") < 0)

    # 2. Key 기반 Inner Join (매칭 쌍 탐색)
    # 점포, POS, 거래번호, 상품코드가 같은 데이터 매칭
    matched_pairs = (
        neg_sales.join(
            pos_sales, 
            on=["점포코드", "POS번호", "거래번호", "상품코드"], 
            suffix="_pos"
        )
        # 금액의 절대값이 같은지 추가 확인 (부분 취소 제외, 완전 취소 타겟)
        .filter(pl.col("판매금액").abs() == pl.col("판매금액_pos"))
        .collect()
    )

    total_neg_count = neg_sales.select(pl.len()).collect().item()
    matched_count = len(matched_pairs)
    match_rate = (matched_count / total_neg_count) * 100

    print(f"전체 음수 데이터 수: {total_neg_count:,} 건")
    print(f"매칭 성공(취소 확인) 데이터 수: {matched_count:,} 건")
    print(f"매칭률(취소율): {match_rate:.2f}%")

    # 3. 시간 차이(Time Gap) 분석
    if matched_count > 0:
        # 판매시간 형식을 초 단위로 변환하여 차이 계산 (HHMMSS)
        # 시간 차이가 0 혹은 매우 짧으면 즉시 취소로 간주
        matched_pairs = matched_pairs.with_columns([
            (pl.col("판매일자") == pl.col("판매일자_pos")).alias("is_same_day")
        ])
        
        same_day_cancel_rate = (len(matched_pairs.filter(pl.col("is_same_day"))) / matched_count) * 100
        print(f"당일 취소 비중: {same_day_cancel_rate:.2f}%")

        if same_day_cancel_rate > 90:
            print("\n[결론 지지] 음수 데이터의 대부분이 당일 취소/무효화 데이터임이 입증되었습니다.")
            print("신상품의 순수 성과 분석 시 판매금액 > 0 인 데이터만 사용해도 통계적 왜곡이 미미할 것으로 판단됩니다.")
        else:
            print("\n[경고] 사후 반품 비중이 높습니다. 단순 제거 대신 기간별 상계 처리(Net Sales)가 권장됩니다.")

    return matched_pairs

In [15]:
validation_results = validate_negative_data(b2_lazy)

--- [검증 시작] 음수 트랜잭션의 정체 및 정당성 분석 ---
전체 음수 데이터 수: 1,030,054 건
매칭 성공(취소 확인) 데이터 수: 585,155 건
매칭률(취소율): 56.81%
당일 취소 비중: 99.63%

[결론 지지] 음수 데이터의 대부분이 당일 취소/무효화 데이터임이 입증되었습니다.
신상품의 순수 성과 분석 시 판매금액 > 0 인 데이터만 사용해도 통계적 왜곡이 미미할 것으로 판단됩니다.


In [21]:
"""
[2차 심층 검증: 매칭되지 않은 음수 데이터(43%)의 영향도 분석]

1. 배경
   - 1차 검증 결과, 음수 데이터의 56.8%만 즉시 취소(Void)로 확인됨.
   - 남은 43.2%가 실제 반품일 경우, 이를 제외하고 분석하는 것이 성공 지표($y$)를 왜곡할 수 있음.

2. 분석 가설
   - 가설 C (거래번호 변경 반품): 매칭되지 않은 데이터 중 상당수는 거래번호가 변경된 채 
     [점포, 상품, 금액]이 일치하는 데이터일 것이다.
   - 가설 D (순위 보존 법칙): 음수 데이터를 포함한 '순매출(Net)'과 양수만 집계한 '총매출(Gross)' 간의 
     상품별 성공 순위(Ranking)는 극도로 유사할 것이다. (상관계수 0.99 이상 예상)

3. 검증 방법론
   - [방법 1] 거래번호를 제외한 유연한 매칭(점포+상품+금액)으로 매칭률 재산출.
   - [방법 2] 상품별로 (양수+음수) 합산 매출과 (양수) 매출을 각각 계산하여 스피어만 상관계수 측정.
"""

def deep_dive_negative_data(b2_lazy):
    print("--- [2차 심층 검증] 음수 데이터 영향도 및 순위 상관관계 분석 ---")
    
    # 데이터 수집 (메모리 효율을 위해 필요한 컬럼만)
    df = b2_lazy.select(["상품코드", "판매금액"]).collect()
    
    # 1. 상품별 Gross Sales (양수만) vs Net Sales (양수+음수) 계산
    prod_stats = df.group_by("상품코드").agg([
        pl.col("판매금액").filter(pl.col("판매금액") > 0).sum().alias("gross_sales"),
        pl.col("판매금액").sum().alias("net_sales")
    ])
    
    # 2. 스피어만 상관계수(Ranking Correlation) 계산
    # 성공 지표는 결국 '누가 더 많이 팔렸나'의 순위가 중요함
    gross_vals = prod_stats["gross_sales"].to_numpy()
    net_vals = prod_stats["net_sales"].to_numpy()
    
    corr, _ = spearmanr(gross_vals, net_vals)
    
    print(f"상품별 Gross vs Net 매출 상관계수: {corr:.6f}")
    
    # 3. 순위 역전 현상 확인 (Top 10000 상품 대상)
    top_gross = prod_stats.sort("gross_sales", descending=True).head(10000)["상품코드"].to_list()
    top_net = prod_stats.sort("net_sales", descending=True).head(10000)["상품코드"].to_list()
    
    mismatch_count = len(set(top_gross) - set(top_net))
    print(f"Top 10000 상품 중 순위권 이탈 발생 수: {mismatch_count}건")

    # 4. 결론 도출 도움
    if corr > 0.99 and mismatch_count <= 5:
        print("\n[검증 결과] 상관계수가 매우 높고 순위 역전이 거의 없습니다.")
        print("음수 데이터를 포함하든 제외하든 '히트 상품'을 선정하는 결론에는 차이가 없음이 입증되었습니다.")
        print("분석의 간결성을 위해 양수(Gross) 데이터만 사용하거나, 안전하게 전체 합산(Net)을 사용해도 무방합니다.")
    else:
        print("\n[주의] 매출 순위 변동이 유의미합니다. 반드시 음수 데이터를 합산(Net Sales)하여 분석해야 합니다.")

    return prod_stats

In [22]:
prod_comparison = deep_dive_negative_data(b2_lazy)

--- [2차 심층 검증] 음수 데이터 영향도 및 순위 상관관계 분석 ---
상품별 Gross vs Net 매출 상관계수: 0.993557
Top 10000 상품 중 순위권 이탈 발생 수: 144건

[주의] 매출 순위 변동이 유의미합니다. 반드시 음수 데이터를 합산(Net Sales)하여 분석해야 합니다.


In [ ]:
# 🧐 2차 검증 결과의 상세 분석 및 비즈니스적 의미

#   1. 왜 상관계수 0.993이 "주의" 대상인가? (가설 D의 재해석)
#    * 가설: 만약 모든 음수 데이터가 모든 상품에 골고루 퍼져 있는 '백색 소음'이라면 상관계수는 1.0에 수렴해야 합니다.
#    * 현실: 0.993이라는 수치는 일부 특정 상품에 음수 매출(반품/취소)이 집중되어 있음을 시사합니다. 예를 들어, 고가의
#      가전제품이나 특정 프로모션으로 대량 구매 후 취소된 상품들은 음수 데이터를 합산하느냐 마느냐에 따라 실적이 수백만
#      원씩 차이 날 수 있습니다.
#    * 결론: "양수만 사용(Gross)"하는 편의주의적 방식은 특정 상품의 성과를 과대평가(Overestimate)할 위험이 있습니다.

#   2. 144건의 순위 역전이 갖는 위험성
#    * 우리의 목표는 HIN 모델을 통해 '진짜 히트 상품'을 맞추는 것입니다. 만약 취소분이 반영되지 않아 실적이 부풀려진
#      상품을 '성공'이라고 라벨링($y=1$)하여 학습시킨다면, 모델은 '잘 팔리는 특징'이 아니라 '취소가 많이 발생하는 특징'을
#      학습하게 되는 치명적인 오류(Data Leakage/Noise)를 범하게 됩니다.

## Preprocessing & NPD Lifecycle Tracking

In [ ]:
# 3. 출시일 정의 및 누적 매출 계산
b2_with_date = b2_lazy.with_columns(pl.col('판매일자').str.to_date('%Y%m%d').alias('sale_dt'))
launch_dates = b2_with_date.group_by('상품코드').agg(pl.col('sale_dt').min().alias('launch_dt'))

sales_enriched = b2_with_date.join(launch_dates, on='상품코드')
sales_enriched = sales_enriched.with_columns((pl.col('sale_dt') - pl.col('launch_dt')).dt.total_days().alias('days_after'))

cum_sales = sales_enriched.group_by('상품코드').agg([
    pl.col('판매금액').filter(pl.col('days_after') <= 7).sum().alias('cum_1w'),
    pl.col('판매금액').filter(pl.col('days_after') <= 14).sum().alias('cum_2w'),
    pl.col('판매금액').filter(pl.col('days_after') <= 28).sum().alias('cum_4w')
]).collect()

print(f"Total Unique Items: {len(cum_sales):,}")